In [1]:
import numpy as np

In [2]:
#Método de Jacobi
def jacobi(A,b,IteMax,erro):
  lenght = len(b)
  #L é triangular inferior, D é matriz diagonal e U é triangular superior
  L = np.zeros((lenght, lenght))
  D = np.zeros((lenght, lenght))
  U = np.zeros((lenght, lenght))

  for i in range(lenght):
    for j in range(lenght):
      if i>j:
        L[i,j] = A[i,j]
      elif i==j:
        D[i,j] = A[i,j]
      else:
        U[i,j] = A[i,j]

  x = np.zeros(lenght)
  x_novo = np.zeros(lenght)
  inversa_D = np.linalg.inv(D)
  erro_absoluto = 1
  for i in range(IteMax):
      x_novo = inversa_D @ (b - (L+U)@x)
      #Usando erro relativo
      erro_relativo = np.linalg.norm(x_novo - x)/np.linalg.norm(x_novo)

      if erro_relativo < erro:
        print(f"Convergiu após {i+1} iterações")
        return x_novo

      x = x_novo
  print("Número de iterações excedidas")
  return x


In [3]:
#Método de Gauss-Seidel
def gauss_seidel(A,b,IteMax,erro):
  lenght = len(b)
  x = np.zeros(lenght)
  x_novo = np.zeros(lenght)
  L_star = np.zeros((lenght,lenght))
  U_star = np.zeros((lenght,lenght))
  Identidade = np.identity(lenght)
  b_star = np.zeros((lenght))

  for i in range (lenght):
    for j in range(lenght):
      if i>j:
        L_star[i,j] = A[i,j]/A[i,i]
      elif i<j:
        U_star[i,j] = A[i,j]/A[i,i]

    b_star[i] = b[i]/A[i,i]

  #(L* + I)**-1
  inv_Le_I = np.linalg.inv(L_star + Identidade)
  for i in range(IteMax):
    x_novo = ((inv_Le_I)*-1) @ U_star @ x + inv_Le_I @ b_star

    erro_at = np.linalg.norm(x_novo-x)/max(np.linalg.norm(x_novo), 1e-15)
    if erro_at<erro:
      print(f"Convergiu com {i+1} iterações")
      return x_novo

    x = x_novo
  print("Número de iterações excedidas")
  return x

In [13]:
#Método de SOR
def SOR(A,b,IteMax,erro,w):
  lenght = len(b)
  L_star = np.zeros((lenght,lenght))
  U_star = np.zeros((lenght,lenght))
  Identidade = np.identity(lenght)
  b_star = np.zeros((lenght))

  A_star = np.zeros((lenght,lenght))
  xGS = np.zeros(lenght)
  xSOR = np.zeros(lenght)
  x = np.zeros(lenght)

  #Inicializando
  for i in range (lenght):
    for j in range(lenght):
      if i>j:
        L_star[i,j] = A[i,j]/A[i,i]
      if i<j:
        U_star[i,j] = A[i,j]/A[i,i]

    b_star[i] = b[i]/A[i,i]


  #Usa o xGS como chute inicial
  inv_Le_I = np.linalg.inv(L_star + Identidade)
  for i in range(lenght):
    x = xGS.copy()
    xGS = ((inv_Le_I) *-1) @ U_star @ xGS + inv_Le_I @ b_star

    erro_absoluto = np.linalg.norm(xGS - x)
    if erro_absoluto<erro:
      break

  #SOR
  xSOR = xGS
  A_star = L_star + Identidade + U_star
  for k in range(IteMax):
    x = xSOR.copy()
    for i in range(lenght):
      soma1=0.0
      for j in range(i):
        soma1 += A_star[i,j]*xSOR[j]

      soma2=0.0
      for j in range(i+1,lenght):
        soma2 += A_star[i,j]*xSOR[j]

      xSOR[i] = (1-w)*xSOR[i] + w*(b_star[i]-soma1-soma2)

    #Erro Relativo
    erro_at = np.linalg.norm(xSOR-x)/max(np.linalg.norm(xSOR),np.linalg.norm(x))
    if erro_at < erro:
      print(f"Convergiu apos {k+1} iterações")
      return xSOR

  print("Número de iterações excedidas")
  return xSOR

In [15]:
def SOR_chute0(A,b,IteMax,erro,w):
  lenght = len(b)
  L_star = np.zeros((lenght,lenght))
  U_star = np.zeros((lenght,lenght))
  Identidade = np.identity(lenght)
  b_star = np.zeros((lenght))

  xSOR = np.zeros(lenght)
  A_star = np.zeros((lenght,lenght))
  x = np.zeros(lenght)

  #Inicializando
  for i in range (lenght):
    for j in range(lenght):
      if i>j:
        L_star[i,j] = A[i,j]/A[i,i]
      if i<j:
        U_star[i,j] = A[i,j]/A[i,i]

    b_star[i] = b[i]/A[i,i]



  #SOR
  #Usando chute inical como 0
  A_star = L_star + Identidade + U_star
  for k in range(IteMax):
    x = xSOR.copy()
    for i in range(lenght):
      soma1=0.0
      for j in range(i):
        soma1 += A_star[i,j]*xSOR[j]

      soma2=0.0
      for j in range(i+1,lenght):
        soma2 += A_star[i,j]*xSOR[j]

      xSOR[i] = (1-w)*xSOR[i] + w*(b_star[i]-soma1-soma2)

      #Erro Relativo
    erro_at = np.linalg.norm(xSOR-x)/max(np.linalg.norm(xSOR),np.linalg.norm(x))
    if erro_at < erro:
      print(f"Convergiu apos {k+1} iterações")
      return xSOR

  print("Número de iterações excedidas")
  return xSOR

In [14]:
#Exercício
matriz_A = np.array([[15,5.,-5],[4,10.,1],[2,-2,8.]])
vetor_b = np.array([30,23,-10.])
erro = 10**-5
maxIte = 30
w1 = 1.
w2 = 1.5
print(jacobi(matriz_A.copy(),vetor_b.copy(),maxIte,erro))
print(gauss_seidel(matriz_A.copy(),vetor_b.copy(),maxIte,erro))
print(f"\nUsando w de {w1}")
print(SOR(matriz_A.copy(),vetor_b.copy(),maxIte,erro,w1))
print(SOR_chute0(matriz_A.copy(),vetor_b.copy(),maxIte,erro,w1))
print(f"\nUsando w de {w2}")
print(SOR(matriz_A.copy(),vetor_b.copy(),maxIte,erro,w2))
print(SOR_chute0(matriz_A.copy(),vetor_b.copy(),maxIte,erro,w2))

Convergiu após 13 iterações
[ 1.0000049   2.00000427 -0.99999981]
Convergiu com 7 iterações
[ 0.99999936  2.00000025 -0.99999978]

Usando w de 1.0
Convergiu apos 4 iterações
[ 0.99999936  2.00000025 -0.99999978]
Convergiu apos 7 iterações
[ 0.99999936  2.00000025 -0.99999978]

Usando w de 1.5
Número de iterações excedidas
[ 0.99998969  2.00003268 -0.99996028]
Número de iterações excedidas
[ 0.99480566  2.01647277 -0.97997928]
